# Approfondissement A1 — Séries temporelles

## Le problème

Un capteur mesure la température d'une salle toutes les heures pendant trois semaines. On veut la moyenne par jour, repérer la tendance malgré le bruit, combler les mesures manquantes, et préparer des variables pour prédire la température de l'heure suivante.

Ces opérations reposent sur un **index temporel** (`DatetimeIndex`) : quand l'index est une date, Pandas sait sélectionner une période, changer de fréquence, calculer des moyennes glissantes ou décaler les valeurs dans le temps.

In [ ]:
import numpy as np
import pandas as pd

capteurs = pd.read_csv("../donnees/demo/capteurs_salle_b12.csv", parse_dates=["horodatage"])
capteurs.head()

## 1. L'index temporel

In [ ]:
serie = capteurs.set_index("horodatage")
serie.index

L'index porte une fréquence (`freq=None` ici, car elle n'a pas été déclarée). On vérifie qu'il est trié et sans doublon :

In [ ]:
serie.index.is_monotonic_increasing, serie.index.is_unique

### Sélectionner une période par une chaîne

Avec un `DatetimeIndex`, `loc` accepte des dates partielles :

In [ ]:
serie.loc["2025-03-05"].shape          # toute la journée du 5 mars

In [ ]:
serie.loc["2025-03-05 08:00":"2025-03-05 11:00"]

## 2. Changer de fréquence : `resample`

`resample` est un `groupby` sur des intervalles de temps. On agrège des mesures horaires en valeurs journalières (*downsampling*) :

In [ ]:
serie["temperature"].resample("D").agg(["min", "mean", "max"]).round(1).head()

### Les alias de fréquence

| Alias | Fréquence |
|---|---|
| `"h"` | heure |
| `"D"` | jour |
| `"W"`, `"W-MON"` | semaine (se terminant le dimanche, le lundi) |
| `"ME"`, `"MS"` | fin de mois, début de mois |
| `"QE"`, `"QS"` | trimestre |
| `"YE"`, `"YS"` | année |
| `"BME"` | dernier jour ouvré du mois |
| `"10YS"` | tous les 10 ans (début d'année) |

Pandas 3 a supprimé plusieurs anciens alias que l'on trouve encore dans les tutoriels et dans les drills :

| Ancien (supprimé) | Actuel |
|---|---|
| `"M"` | `"ME"` |
| `"BM"` | `"BME"` |
| `"A"`, `"Y"` (fin d'année) | `"YE"` |
| `"AS"`, `"AS-JAN"` | `"YS"` |
| `"H"` | `"h"` |

In [ ]:
serie["temperature"].resample("M").mean()

In [ ]:
serie["temperature"].resample("W").mean().round(2)

### `to_period` : regrouper par période calendaire

Une alternative à `resample` est de convertir chaque date en période (mois, semaine, année), puis de grouper :

In [ ]:
serie["temperature"].groupby(serie.index.to_period("W")).mean().round(2)

## 3. Combler les trous

In [ ]:
serie["temperature"].isna().sum()

In [ ]:
extrait = serie.loc["2025-03-01", "temperature"]
pd.DataFrame({
    "brut": extrait,
    "ffill": extrait.ffill(),                      # valeur précédente
    "interpolate": extrait.interpolate(),         # interpolation linéaire
}).loc[extrait.isna() | extrait.shift(1).isna() | extrait.shift(-1).isna()].round(2)

Pour une température, l'interpolation linéaire est raisonnable. Pour un stock ou un prix affiché, reprendre la dernière valeur connue (`ffill`) a plus de sens. Le choix dépend de ce que représente la variable.

## 4. Moyenne mobile : `rolling`

Une moyenne mobile remplace chaque valeur par la moyenne des `n` dernières : elle lisse le bruit et fait ressortir la tendance.

In [ ]:
lisse = pd.DataFrame({
    "brut": serie["temperature"],
    "moyenne_6h": serie["temperature"].rolling(6).mean(),
    "moyenne_24h": serie["temperature"].rolling(24, min_periods=12).mean(),
})
lisse.iloc[20:30].round(2)

> **Prédiction.** Pourquoi les 5 premières valeurs de `moyenne_6h` sont-elles `NaN` ? Que change `min_periods` ?

Une fenêtre peut aussi s'exprimer en durée (`rolling("24h")`), ce qui reste correct même si des mesures manquent dans l'index.

In [ ]:
serie["temperature"].rolling("24h").mean().iloc[[0, 1, 23, 24]].round(2)

## 5. Décaler dans le temps : `shift` et `diff`

Pour prédire la température de l'heure suivante, les variables les plus utiles sont souvent les valeurs **passées** : la mesure d'il y a 1 heure, d'il y a 24 heures.

In [ ]:
features = pd.DataFrame({
    "temperature": serie["temperature"],
    "lag_1h": serie["temperature"].shift(1),
    "lag_24h": serie["temperature"].shift(24),
    "variation_1h": serie["temperature"].diff(),
    "heure": serie.index.hour,
    "jour_semaine": serie.index.day_of_week,
})
features.iloc[24:29].round(2)

`shift(1)` décale les valeurs d'un pas vers le bas : chaque ligne reçoit la valeur précédente. Une variable construite avec `shift(-1)` (la valeur **suivante**) introduirait une fuite de données dans un modèle de prévision.

## 6. Profil moyen : grouper par une composante de la date

In [ ]:
profil = serie.groupby(serie.index.hour)["temperature"].mean().round(2)
profil.loc[[3, 9, 15, 21]]

## 7. Une série financière

Les cours de l'action Apple (fichier des drills) ont une ligne par jour de bourse, du plus récent au plus ancien.

In [ ]:
apple = pd.read_csv("../donnees/drills/appl_1980_2014.csv", parse_dates=["Date"]).set_index("Date")
apple.index.is_monotonic_increasing

In [ ]:
apple = apple.sort_index()
mensuel = apple["Adj Close"].resample("BME").last()        # dernier jour ouvré de chaque mois
mensuel.tail(3)

In [ ]:
rendement = apple["Adj Close"].pct_change()                # variation relative d'un jour à l'autre
rendement.describe().round(4)

## 8. Construire une date à partir de colonnes séparées

Le fichier `wind.data` (drill D06) stocke l'année sur deux chiffres, le mois et le jour dans trois colonnes. L'ancienne écriture `parse_dates=[[0, 1, 2]]` a été supprimée en Pandas 3 ; on construit la date avec `pd.to_datetime` à partir d'un DataFrame de colonnes `year`, `month`, `day`.

In [ ]:
vent = pd.read_csv("../donnees/drills/wind.data", sep=r"\s+")
vent.head(2)

In [ ]:
dates = pd.to_datetime(pd.DataFrame({"year": vent["Yr"] + 1900, "month": vent["Mo"], "day": vent["Dy"]}))
vent = vent.drop(columns=["Yr", "Mo", "Dy"]).set_index(dates)
vent.index[[0, -1]]

Ajouter 1900 règle aussi le problème du siècle que le drill fait corriger à l'étape 4 : une année écrite `61` interprétée automatiquement deviendrait 2061.

## Exercices

- Drills **D04 US Crime Rates**, **D06 Wind Stats**, **D08 Apple Stock** : les solutions d'origine utilisent des alias supprimés (`10AS`, `BM`, `A`) ou `parse_dates=[[...]]`. Des solutions compatibles avec Pandas 3 sont fournies dans `corrections/drills/pandas3/`.

## À retenir

- Un `DatetimeIndex` permet de sélectionner par date partielle : `loc["2025-03"]`.
- `resample("D")` change de fréquence ; `to_period` regroupe par période calendaire.
- Alias actuels : `h`, `D`, `W`, `ME`, `BME`, `QE`, `YE`, `YS`.
- `ffill` et `interpolate` comblent les trous, selon le sens de la variable.
- `rolling` calcule des moyennes mobiles ; `shift` et `diff` créent des variables décalées, toujours vers le passé.